# 09_generate_dev_seed — dev 합성 seed 생성

Databricks workspace에 bundle deploy된 `data/dev/generator/generate.py`를 실행해 Landing/Eval/Context JSONL을 생성한다.

In [0]:
dbutils.widgets.text("seed_data_path", "data/dev/landing/raw")
dbutils.widgets.text("eval_seed_path", "data/dev/eval/ground_truth_case.jsonl")
dbutils.widgets.text("context_output_root", "data/dev/context")
dbutils.widgets.text("dev_seed_size", "20")
dbutils.widgets.text("dev_seed", "35")
dbutils.widgets.text("dev_reference_date", "2026-06-29")
dbutils.widgets.text("generate_dev_seed_enabled", "true")

SEED_DATA_PATH = dbutils.widgets.get("seed_data_path").strip("/")
EVAL_SEED_PATH = dbutils.widgets.get("eval_seed_path").strip("/")
CONTEXT_OUTPUT_ROOT = dbutils.widgets.get("context_output_root").strip("/")
DEV_SEED_SIZE = dbutils.widgets.get("dev_seed_size")
DEV_SEED = dbutils.widgets.get("dev_seed")
DEV_REFERENCE_DATE = dbutils.widgets.get("dev_reference_date")
GENERATE_DEV_SEED_ENABLED = dbutils.widgets.get("generate_dev_seed_enabled").strip().lower() == "true"

WORKSPACE_FILES_ROOT = "/Workspace/Shared/access-drift/files"
GENERATOR_ROOT = f"{WORKSPACE_FILES_ROOT}/data/dev/generator"
LANDING_OUTPUT_ROOT = f"{WORKSPACE_FILES_ROOT}/{SEED_DATA_PATH}"
EVAL_OUTPUT_PATH = f"{WORKSPACE_FILES_ROOT}/{EVAL_SEED_PATH}"
CONTEXT_OUTPUT_PATH = f"{WORKSPACE_FILES_ROOT}/{CONTEXT_OUTPUT_ROOT}"

if not GENERATE_DEV_SEED_ENABLED:
    dbutils.notebook.exit("generate_dev_seed_task skipped")


In [0]:
import os
import subprocess
import sys
from pathlib import Path

generate_py = Path(GENERATOR_ROOT) / "generate.py"
requirements = Path(GENERATOR_ROOT) / "requirements.txt"

if not generate_py.is_file():
    raise RuntimeError(f"Generator script not found: {generate_py}")
if not requirements.is_file():
    raise RuntimeError(f"Generator requirements not found: {requirements}")

print(f"generator OK: {generate_py}")


In [0]:
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(requirements)])

cmd = [
    sys.executable,
    str(generate_py),
    "--size", DEV_SEED_SIZE,
    "--seed", DEV_SEED,
    "--reference-date", DEV_REFERENCE_DATE,
    "--landing-output-root", LANDING_OUTPUT_ROOT,
    "--eval-output-path", EVAL_OUTPUT_PATH,
    "--context-output-root", CONTEXT_OUTPUT_PATH,
]
print("running:", " ".join(cmd))
subprocess.check_call(cmd)


In [0]:
def count_jsonl(path: Path) -> int:
    if not path.is_file():
        raise RuntimeError(f"generated file not found: {path}")
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

landing_files = [
    Path(LANDING_OUTPUT_ROOT) / "azure/applications.jsonl",
    Path(LANDING_OUTPUT_ROOT) / "azure/application_owners.jsonl",
    Path(LANDING_OUTPUT_ROOT) / "azure/password_credentials.jsonl",
    Path(LANDING_OUTPUT_ROOT) / "identity/lifecycle_seed.jsonl",
    Path(LANDING_OUTPUT_ROOT) / "assets/assets_seed.jsonl",
    Path(LANDING_OUTPUT_ROOT) / "access/access_edges_seed.jsonl",
    Path(LANDING_OUTPUT_ROOT) / "github/apps.jsonl",
    Path(LANDING_OUTPUT_ROOT) / "github/credentials.jsonl",
    Path(LANDING_OUTPUT_ROOT) / "github/repos.jsonl",
    Path(LANDING_OUTPUT_ROOT) / "github/access_edges.jsonl",
    Path(LANDING_OUTPUT_ROOT) / "google_workspace/users.jsonl",
    Path(LANDING_OUTPUT_ROOT) / "google_workspace/assets.jsonl",
    Path(LANDING_OUTPUT_ROOT) / "google_workspace/access_edges.jsonl",
    Path(LANDING_OUTPUT_ROOT) / "slack/users.jsonl",
    Path(LANDING_OUTPUT_ROOT) / "slack/channels.jsonl",
    Path(LANDING_OUTPUT_ROOT) / "slack/access_edges.jsonl",
]
context_files = [
    Path(CONTEXT_OUTPUT_PATH) / "asset_context.jsonl",
    Path(CONTEXT_OUTPUT_PATH) / "assignment_routing.jsonl",
]

for path in landing_files + [Path(EVAL_OUTPUT_PATH)] + context_files:
    rows = count_jsonl(path)
    if rows <= 0:
        raise RuntimeError(f"generated file has no rows: {path}")
    print(f"OK {path}: {rows} rows")
